### Structured output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.


In [2]:
import os
from langchain_groq import ChatGroq

model = ChatGroq(
    model = "llama-3.3-70b-versatile"
)

In [3]:
from pydantic import BaseModel, Field
from typing import List,Any

class Song(BaseModel):
    title : str = Field(description="Title of the Song")
    singer : List[str] = Field(description="Singers of the song")
    year: str = Field(description="Year of release of the song")
    movie_or_album: str = Field(description="Movie or Album of the song")

model_with_structured_output = model.with_structured_output(Song)
response = model_with_structured_output.invoke("Give me the details related to the song Phir Mohabbat")
response

Song(title='Phir Mohabbat', singer=['Arijit Singh', 'Mohammed Irfan'], year='2011', movie_or_album='Murder 2')

### Message Output alongside Parsed Structured

In [6]:
class Song(BaseModel):
    title : str = Field(..., description="Title of the Song")
    singer : List[str] = Field(..., description="Singers of the song")
    year: str = Field(..., description="Year of release of the song")
    movie_or_album: str = Field(..., description="Movie or Album of the song")

model_with_structured_output = model.with_structured_output(Song, include_raw=True)
response = model_with_structured_output.invoke("Give me the details related to the song Tu hi meri shab hai")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'j8mcgsq8t', 'function': {'arguments': '{"movie_or_album":"Gangster","singer":["K.K."],"title":"Tu Hi Meri Shab Hai","year":"2005"}', 'name': 'Song'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 297, 'total_tokens': 339, 'completion_time': 0.168647821, 'completion_tokens_details': None, 'prompt_time': 0.030055951, 'prompt_tokens_details': None, 'queue_time': 0.051771359, 'total_time': 0.198703772}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fc2bf-831d-7200-b6ee-1e8d9c0ee036-0', tool_calls=[{'name': 'Song', 'args': {'movie_or_album': 'Gangster', 'singer': ['K.K.'], 'title': 'Tu Hi Meri Shab Hai', 'year': '2005'}, 'id': 'j8mcgsq8t', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens':

### Nested Structure

In [12]:
class Singer(BaseModel):
    name : str = Field(description="Name of the Singer")
    age : int = Field(description="Age of the singer")
    hometown : str = Field(description="HomeTown of the Singer")

class Song(BaseModel):
    title : str = Field(description="Title of the Song")
    singer : List[Singer] = Field(description="Singers of the song")
    year: str = Field(description="Year of release of the song")
    movie_or_album: str = Field(description="Movie or Album of the song")

model_with_structured_output = model.with_structured_output(Song)
response = model_with_structured_output.invoke("Give me the details related to the song Tu Hi meri shab hai")
response

Song(title='Tu Hi Meri Shab Hai', singer=[Singer(name='K.K.', age=54, hometown='Delhi')], year='2006', movie_or_album='Gangster')

### TypedDict

TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.


In [23]:
from typing_extensions import TypedDict, Annotated

class Car(TypedDict):
    brand: Annotated[str, ..., "Brand name of the car"]
    model: Annotated[str, ..., "Model name of the car"]
    fuel_type: Annotated[str, ..., "Fuel Type of the Car [Petrol/Diesel/Electric]"]
    max_speed: Annotated[float, ..., "Gives the Maximum Speed of the car in KM/hr"]
    year: Annotated[int, ..., "year of release of the model"]
    mileage: Annotated[float, ..., "mileage of the car model"]
    price: Annotated[float, ..., "Price of the car in lakhs INR"]

st_model = model.with_structured_output(Car)
response = st_model.invoke("Give Me the details about the Car BMW M4")
response

{'brand': 'BMW',
 'fuel_type': 'Petrol',
 'max_speed': 155,
 'mileage': 10.6,
 'model': 'M4',
 'price': 81.4,
 'year': 2021}

### Data Class

In [25]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class contactInfo:
    name : str
    phone : str
    email : str


agents = create_agent(
    model = model,
    response_format=contactInfo
)


response = agents.invoke(
    {
        "messages" : [
            {
                "role" : "user",
                "content" : "Extract the contact info from : Shubman Gill, shubman77@bcci.in, (+91) 8239913772"
            }
        ]
    }
)

response['structured_response']

contactInfo(name='Shubman Gill', phone='(+91) 8239913772', email='shubman77@bcci.in')